# P/PL — pretrained ConvNeXt-Tiny with vs without frozen Stage 3/4 priors (protocol-locked)

**Question (PL − P, primary).** Do frozen vessel/lesion segmentation-derived priors add information when
supplied to the **same** strong pretrained CNN? **PL − frozen NO_RACAF** is a system-level comparison only;
it does **not** isolate pretraining. P/PL is **not** a novel architecture.

| arm | input to ConvNeXt-Tiny (ImageNet-1k) | first-layer kernel | head |
|---|---|---|---|
| **P** | RGB (Stage-02, ImageNet-normalised) | pretrained 4×4×3×96 | 768 → CORN Dense(768→4) |
| **PL** | RGB + vessel + MA + HE + EX + SE (priors unnormalised, [0,1]) | [pretrained RGB \| zeros] 4×4×8×96 | identical, same seeded init |

Seeds 42 / 123 / 2026, 6 runs, each once. Weighted CORN, AdamW lr 1e-4 wd 0.05 (no decay on any 1-D
parameter, incl. LayerScale), ReduceLROnPlateau, EarlyStopping, ≤ 50 epochs, batch 2, mixed_float16, BEST by
val_QWK, LAST also evaluated. Primary endpoint: AUROC of P(grade ≥ 3), grade 4 vs 0–2. **DDR is not used.**

**How to use.** Select a T4 GPU runtime and *Run all*. The pinned ImageNet-1k weights
(`convnext_tiny_notop.h5`, SHA-256 `d547c096…`) are downloaded automatically once and cached on Drive. The
notebook freezes `PREREGISTRATION.json`, runs **all protocol pre-flight assertions** (it stops before any
training if one fails), then trains/evaluates the first unfinished run and continues through all six.
After a disconnect, *Run all* again — completed runs are skipped and interrupted ones resume. Nothing is
edited by hand between sessions. Implementation: `pl_convnext.py`.

In [ ]:
# ==== [1] ENVIRONMENT / IMPORTS ====
import os
import posixpath
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import setup as colab_setup
setup_info = colab_setup.setup()

import colab_config
import verify_environment
env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR, drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"), require_gpu=True)

import csv
import datetime
import gc
import json
import time

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

import config
import icdr_two_route_experiment as c1
import improved_training_data as itd
import local_feature_extraction_dataset as lfed
import multiseed_runs as msr
import pl_convnext as pl
import weighted_corn
from lesion_segmentation_dataset import LESION_CLASSES
from training import (Trainer, TrainingConfig, CheckpointOptions, TrainingStateCheckpoint,
                      enable_mixed_precision, expected_policy_name, model_precision_policies)
from training import checkpointing as ckpt
from training.trainer import precision_is_consistent

GPUS = tf.config.list_physical_devices("GPU")
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
print(f"Ready. P/PL {pl.EXPERIMENT_VERSION}; keras {keras.__version__}; tensorflow {tf.__version__}; "
      f"repo {REPO_COMMIT}; GPUs {GPUS}")

In [ ]:
# ==== [2] CONFIGURATION AND FROZEN PRE-REGISTRATION ====
# Written BEFORE any training. On every later session it is re-verified; any difference stops the notebook.
EXPERIMENTS_ROOT = colab_config.DRIVE.experiments_root
SIX_RUN_ROOT = posixpath.join(EXPERIMENTS_ROOT, "ImprovedTraining")
SIX_RUN_ID = "improved_multiseed_2026_09"
SIX_RUN_DIR = msr.experiment_root(SIX_RUN_ROOT, SIX_RUN_ID)
PARENT_DIR = posixpath.join(EXPERIMENTS_ROOT, pl.EXPERIMENT_NAME)
WEIGHTS_CACHE_DIR = posixpath.join(PARENT_DIR, "pretrained_weights")

os.makedirs(PARENT_DIR, exist_ok=True)
_existing = sorted(d for d in os.listdir(PARENT_DIR)
                   if os.path.exists(posixpath.join(PARENT_DIR, d, "PREREGISTRATION.json")))
if len(_existing) > 1:
    raise RuntimeError(f"More than one pre-registered experiment under {PARENT_DIR}: {_existing}.")
EXPERIMENT_TIMESTAMP = _existing[0] if _existing else datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
EXPERIMENT_ID = f"pl_convnext_{EXPERIMENT_TIMESTAMP}"
EXPERIMENT_DIR = posixpath.join(PARENT_DIR, EXPERIMENT_TIMESTAMP)
for _protected in (SIX_RUN_DIR, posixpath.join(EXPERIMENTS_ROOT, "Grade4AuxLoss"),
                   posixpath.join(EXPERIMENTS_ROOT, "Task3WeightingPlacebo"),
                   posixpath.join(EXPERIMENTS_ROOT, "C1_ICDRTwoRouteHead"),
                   posixpath.join(EXPERIMENTS_ROOT, "Grade3vs4_Phase0"),
                   colab_config.DRIVE.experiment_dir("FinalClassification")):
    assert posixpath.commonpath([posixpath.normpath(EXPERIMENT_DIR), posixpath.normpath(_protected)]) \
        != posixpath.normpath(_protected), f"{EXPERIMENT_DIR} is under protected {_protected}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

SIX_RUN_PREREG, _ = msr.load_and_verify_preregistration(
    posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
BATCH_SIZE = SIX_RUN_PREREG["batch_size"]
MAX_EPOCHS = SIX_RUN_PREREG["max_epochs"]
LEARNING_RATE = SIX_RUN_PREREG["learning_rate"]
WEIGHT_DECAY = SIX_RUN_PREREG["weight_decay"]
CLASS_WEIGHTS = list(SIX_RUN_PREREG["class_weights"])
MONITOR, MODE = SIX_RUN_PREREG["primary_metric"], "max"
EARLY_STOPPING_PATIENCE = SIX_RUN_PREREG["early_stopping"]["patience"]
REDUCE_LR_PATIENCE = SIX_RUN_PREREG["reduce_lr_on_plateau"]["patience"]
REDUCE_LR_FACTOR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["factor"]
MIN_LR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["min_lr"]
_locked = {"BATCH_SIZE": (BATCH_SIZE, 2), "MAX_EPOCHS": (MAX_EPOCHS, 50), "LEARNING_RATE": (LEARNING_RATE, 1e-4),
           "WEIGHT_DECAY": (WEIGHT_DECAY, 0.05), "MONITOR": (MONITOR, "val_QWK"),
           "EARLY_STOPPING_PATIENCE": (EARLY_STOPPING_PATIENCE, 12), "REDUCE_LR_PATIENCE": (REDUCE_LR_PATIENCE, 4),
           "REDUCE_LR_FACTOR": (REDUCE_LR_FACTOR, 0.5), "MIN_LR": (MIN_LR, 1e-6)}
_bad = {k: v for k, v in _locked.items() if v[0] != v[1]}
assert not _bad, f"six-run protocol values differ from the locked P/PL protocol: {_bad}"
assert np.allclose(CLASS_WEIGHTS, weighted_corn.PREREGISTERED_CLASS_WEIGHTS)
with open(posixpath.join(SIX_RUN_DIR, "experiment_manifest.json")) as _fh:
    SIX_RUN_POPULATION = json.load(_fh)
EXPECTED_TRAIN_N, EXPECTED_VAL_N = 2921, 730

PL_MODULE_SOURCES = (("pl_convnext.py", msr.WHOLE_MODULE),)
PL_MODULE_DIGEST = msr.normalized_source_digests("pl_convnext.py", msr.WHOLE_MODULE, msr.REPO_ROOT)

PREREGISTRATION = {
    "experiment_name": pl.EXPERIMENT_NAME, "experiment_id": EXPERIMENT_ID,
    "experiment_version": pl.EXPERIMENT_VERSION,
    "question_primary_A": ("Do frozen vessel/lesion segmentation-derived priors add useful information when "
                           "supplied to the same strong pretrained CNN? (PL - P)"),
    "question_system_B": ("PL vs the frozen improved_multiseed_2026_09 NO_RACAF BEST reference: a system-level "
                          "comparison only; it does not isolate pretraining."),
    "novelty": "none claimed: P/PL is not a novel architecture",
    "backbone": {"model": "keras.applications.ConvNeXtTiny", "weights_url": pl.WEIGHTS_URL,
                 "weights_sha256": pl.WEIGHTS_SHA256, "pretraining": "ImageNet-1k only",
                 "include_top": False, "include_preprocessing": False, "pooling": "avg",
                 "drop_path_rate": 0.0, "feature_dim": pl.FEATURE_DIM, "image_size": pl.IMAGE_SIZE},
    "arms": {"P": "RGB only (channels 0-2 of stage5_input)",
             "PL": "RGB + vessel + MA + HE + EX + SE (channels 0-7, in this order)"},
    "channel_order": list(pl.CHANNEL_ORDER),
    "input_normalisation": {"rgb": {"mean": list(pl.IMAGENET_MEAN), "std": list(pl.IMAGENET_STD),
                                    "note": "fixed constants, identical to Keras PreStem on x*255"},
                            "priors": "identity (cached probabilities in [0,1]); no data statistics"},
    "first_layer": ("PL kernel[:, :, 0:3, :] = pretrained RGB kernel exactly; kernel[:, :, 3:8, :] = 0; "
                    "bias copied unchanged; all other weights copied positionally with shape checks"),
    "head": "CORN Dense(768->4); kernel GlorotUniform(seed=run_seed), bias zeros; identical for P and PL",
    "optimizer": {"name": "AdamW", "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                  "no_decay": "every 1-D trainable parameter (biases, LayerNorm, LayerScale) + names "
                              + str(list(msr.WEIGHT_DECAY_EXCLUDE_NAMES))},
    "training": {"loss": "weighted_corn (PREREGISTERED_CLASS_WEIGHTS)", "class_weights": CLASS_WEIGHTS,
                 "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "monitor": MONITOR, "mode": MODE,
                 "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                 "reduce_lr": [REDUCE_LR_PATIENCE, REDUCE_LR_FACTOR, MIN_LR], "mixed_precision": True,
                 "augmentation": "unchanged (flips + rot90 on all 8 channels; RGB brightness/contrast +-10%)",
                 "checkpoint": "BEST by val_QWK; LAST also evaluated", "split_sha256": msr.EXPECTED_SPLIT_SHA256,
                 "population": [EXPECTED_TRAIN_N, EXPECTED_VAL_N], "no_cache_regeneration": True},
    "seeds": list(pl.SEEDS), "run_order": [f"{a}-{s}" for a, s in pl.RUN_ORDER],
    "primary_endpoints": {"A": "per-seed PL - P AUROC of P(grade>=3), grade 4 vs 0-2 (BEST)",
                          "B": "per-seed PL - frozen NO_RACAF BEST, same endpoint (system-level)"},
    "secondary": ["AUROC bootstrap 95% CI (C1 procedure: 2000 stratified paired resamples, seed 20260927)",
                  "grade-4 sensitivity at 95% specificity among grades 0-2", "QWK", "MAE",
                  "grade-3 recall", "false-urgent rate", "all metrics at LAST"],
    "decision_rule": {"SUPPORTIVE": "mean >= +0.03 AND > 0 in 3/3 seeds AND guardrails hold",
                      "NOT_SUPPORTIVE": "mean < +0.01 OR <= 0 in >= 2/3 seeds",
                      "INCONCLUSIVE": "otherwise (including a met primary with a failed guardrail)",
                      "guardrails_on_3_seed_mean": {k: list(v) for k, v in pl.GUARDRAILS.items()},
                      "applied_independently_to": ["A", "B"]},
    "excluded": ["persistent-21 set as an endpoint", "DDR (reserved; separate audit first)",
                 "any threshold, seed or architecture selection", "re-running any seed"],
    "pl_module_source_digest": PL_MODULE_DIGEST,
}
PREREG_PATH = posixpath.join(EXPERIMENT_DIR, "PREREGISTRATION.json")
_expected = json.loads(json.dumps(PREREGISTRATION))
if os.path.exists(PREREG_PATH):
    _stored, _ = msr.load_and_verify_preregistration(PREREG_PATH)
    if {k: v for k, v in _stored.items() if k != "frozen_on"} != _expected:
        raise RuntimeError(f"{PREREG_PATH} differs from this notebook's pre-registration (or pl_convnext.py "
                           "changed). The protocol is frozen.")
    print(f"Pre-registration verified (frozen on {_stored.get('frozen_on')}): {PREREG_PATH}")
else:
    with open(PREREG_PATH, "w") as _fh:
        _fh.write(json.dumps(dict(_expected, frozen_on=datetime.datetime.now().isoformat(timespec="seconds")),
                             indent=2, sort_keys=True))
    print(f"Pre-registration FROZEN now at {PREREG_PATH}")

In [ ]:
# ==== [3] DATA, CACHE AND PINNED PRETRAINED WEIGHTS ====
# The existing cache archive is only EXTRACTED; no cache is regenerated. Stage 03/04 models are never loaded.
TRAIN_ENTRIES, VAL_ENTRIES, SPLIT_SHA256 = msr.verify_split()
LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
LOCAL_CACHE_MARKER = "/content/cache/.multiseed_archive_extracted.json"
CACHE_ARCHIVE_DIR = os.path.join(os.path.dirname(config.LOCAL_FEATURE_RESULTS_DIR), "cache_archive")
STAGING_DIR = "/content/checkpoint_staging_pl"
STAGE34_PATHS = [config.VESSEL_SEG_MODEL_DIR, config.LESION_SEG_MODEL_DIR, CACHE_ARCHIVE_DIR]
DATA = {"ready": False, "train": None, "val": None}


def ensure_local_cache():
    if DATA["ready"]:
        return
    if not os.path.exists(LOCAL_CACHE_MARKER):
        import joint_cache_archive as jca
        plan = jca.plan_extraction(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                   racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        jca.print_extraction_plan(plan)
        if plan["drive_unreachable"] or not plan["fits"]:
            raise RuntimeError("Refusing to extract the cache archive.")
        extract = jca.extract_archive(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR, min_free_bytes=plan["required_bytes"])
        jca.print_extract(extract)
        if extract["corrupt"] or extract["drive_unreachable"]:
            raise RuntimeError("Archive extraction did not complete. Re-run the notebook.")
        with open(LOCAL_CACHE_MARKER, "w") as fh:
            json.dump({"extracted": datetime.datetime.now().isoformat(timespec="seconds")}, fh)
    DATA["train"] = itd.locally_cached_entries(TRAIN_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    DATA["val"] = itd.locally_cached_entries(VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    if (len(DATA["train"]), len(DATA["val"])) != (int(SIX_RUN_POPULATION["n_train_yielded"]),
                                                   int(SIX_RUN_POPULATION["n_val_yielded"])):
        raise RuntimeError("Extracted cache population differs from the six-run pin. No cache is regenerated.")
    DATA["ready"] = True


def fresh_session():
    gc.collect()
    tf.keras.backend.clear_session()


ensure_local_cache()
print(f"Population: {len(DATA['train'])} train / {len(DATA['val'])} val; split sha256 {SPLIT_SHA256}")

WEIGHTS = pl.ensure_pretrained_weights(WEIGHTS_CACHE_DIR)
print("Pretrained weights:", json.dumps({k: WEIGHTS[k] for k in ("source_url", "sha256", "obtained_via",
                                                                  "drive_cache")}))
enable_mixed_precision(False)                       # reference arrays are taken in float32
fresh_session()
REFERENCE, REFERENCE_ARRAYS = pl.load_reference(WEIGHTS["local_path"])
print(f"Reference ConvNeXt-Tiny loaded from the pinned file: {len(REFERENCE_ARRAYS)} arrays")

_fp_path = posixpath.join(EXPERIMENT_DIR, "stage34_fingerprint_initial.json")
STAGE34_FINGERPRINT = pl.directory_fingerprint(STAGE34_PATHS)
if not os.path.exists(_fp_path):
    with open(_fp_path, "w") as _fh:
        json.dump({"recorded": datetime.datetime.now().isoformat(timespec="seconds"),
                   "fingerprint": STAGE34_FINGERPRINT}, _fh, indent=2)
with open(_fp_path) as _fh:
    STAGE34_FINGERPRINT_INITIAL = json.load(_fh)["fingerprint"]

In [ ]:
# ==== [4] PROTOCOL PRE-FLIGHT ASSERTIONS (every session, BEFORE any training) ====
PREFLIGHT = {}


def check(number, name, ok, detail):
    PREFLIGHT[f"{number:02d} {name}"] = {"ok": bool(ok), "detail": detail}
    print(f"  [{'PASS' if ok else 'FAIL'}] {number:2d}. {name}: {detail}")


def cached_sample(entry):
    return itd.load_cached_sample(entry[0], entry[1], LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR, False, None)


print("Pre-flight:")
check(1, "Colab GPU available", bool(GPUS), [g.name for g in GPUS])
_paths = {"drive_mount": colab_config.DRIVE_MOUNT_POINT, "experiments_root": EXPERIMENTS_ROOT,
          "six_run_dir": SIX_RUN_DIR, "cache_archive": CACHE_ARCHIVE_DIR,
          **{f"reference_seed_{s}": posixpath.join(msr.run_dir(SIX_RUN_ROOT, SIX_RUN_ID, "NO_RACAF", s),
                                                    "evaluation", "per_sample_best.csv") for s in pl.SEEDS}}
_missing = [k for k, p in _paths.items() if not os.path.exists(p)]
check(2, "Drive mounted and project paths exist", not _missing, _missing or "all present")
check(3, "Pinned ImageNet ConvNeXt weights load", WEIGHTS["sha256"] == pl.WEIGHTS_SHA256
      and len(REFERENCE_ARRAYS) == len(pl.build_backbone("P").weights),
      f"sha256 {WEIGHTS['sha256'][:16]}..., {len(REFERENCE_ARRAYS)} arrays aligned")

# 4-7, 12: structure and numerics in float32, every seed
_samples = [cached_sample(e) for e in DATA["val"][:2]]
_s5 = np.stack([s["stage5_input"] for s in _samples]).astype("float32")
_s6 = np.stack([s["stage6_input"] for s in _samples]).astype("float32")
_rel = np.array([s["reliability"] for s in _samples], np.float32).reshape(-1, 1)
_ok4, _ok5, _ok6, _ok7, _ok12, _max7, _max_ref, _counts = True, True, True, True, True, 0.0, 0.0, {}
_ref_stem = [a for a in REFERENCE_ARRAYS if a.ndim == 4 and a.shape[:3] == (4, 4, 3)][0]
_ref_feat = REFERENCE(_s5[..., :3] * 255.0).numpy()     # Keras' own preprocessing path, computed once
for _seed in pl.SEEDS:
    enable_mixed_precision(False)
    fresh_session()
    _p = pl.build_pl_model("P", _seed, REFERENCE_ARRAYS)
    _l = pl.build_pl_model("PL", _seed, REFERENCE_ARRAYS)
    _bp, _bl = _p.get_layer(pl.BACKBONE_NAME), _l.get_layer(pl.BACKBONE_NAME)
    _ok4 &= all(np.array_equal(v.numpy(), a) for v, a in zip(_bp.weights, REFERENCE_ARRAYS))
    _k = pl.stem_kernel(_bl).numpy()
    _ok5 &= bool(np.array_equal(_k[:, :, :3, :], _ref_stem)) and all(
        np.array_equal(v.numpy(), a) for v, a in zip(_bl.weights, REFERENCE_ARRAYS) if v.shape == a.shape)
    _ok6 &= _k.shape == (4, 4, 8, 96) and not _k[:, :, 3:, :].any()
    _lp = _p.predict([_s5, _s6, _rel], verbose=0)
    _ll = _l.predict([_s5, _s6, _rel], verbose=0)
    _max7 = max(_max7, float(np.max(np.abs(_lp - _ll))))
    _feat = _bp(_p.get_layer("channel_adapter_p")(_s5)).numpy()
    _max_ref = max(_max_ref, float(np.max(np.abs(_feat - _ref_feat))))
    _hk, _hb = pl.corn_head_initial_weights(_seed)
    for _m in (_p, _l):
        _w = _m.get_layer("corn").get_layer("corn_logits").get_weights()
        _ok12 &= bool(np.array_equal(_w[0], _hk) and np.array_equal(_w[1], _hb))
    _counts[_seed] = {"P": pl.trainable_parameter_count(_p), "PL": pl.trainable_parameter_count(_l)}
    del _p, _l, _bp, _bl
_counts_ok = all(c["P"] == pl.EXPECTED_BACKBONE_PARAMETERS["P"] + pl.EXPECTED_HEAD_PARAMETERS
                 and c["PL"] == pl.EXPECTED_BACKBONE_PARAMETERS["PL"] + pl.EXPECTED_HEAD_PARAMETERS
                 for c in _counts.values())
check(4, "P weights match the pretrained reference", _ok4 and _max_ref <= pl.REFERENCE_PARITY_TOL and _counts_ok,
      f"all arrays equal; |P(norm x) - R(x*255)| max {_max_ref:.2e} (tol {pl.REFERENCE_PARITY_TOL}); "
      f"parameters {_counts[42]}")
check(5, "PL RGB first-layer slice equals the pretrained kernel", _ok5, "exact, all other arrays equal")
check(6, "PL extra five input-channel kernels are zero", _ok6, "kernel (4, 4, 8, 96), [:, :, 3:, :] == 0")
check(7, "Initial PL(x8) == P(xRGB)", _max7 <= pl.PL_P_EQUIVALENCE_TOL,
      f"max |dlogit| {_max7:.2e} (tol {pl.PL_P_EQUIVALENCE_TOL}, float32, 3 seeds, real cached samples)")

_derived = lfed._resize_input(_samples[0]["stage5_input"][..., :3], itd.jtd.STAGE6_IMAGE_SIZE)
check(8, "Channel order RGB, vessel, MA, HE, EX, SE",
      tuple(LESION_CLASSES) == pl.EXPECTED_LESION_CLASSES and lfed.NUM_CHANNELS == 8
      and np.allclose(_derived, _samples[0]["stage6_input"], atol=1e-6),
      f"LESION_CLASSES {tuple(LESION_CLASSES)}; channels 0-2 are the RGB Stage 06 is derived from")

_scan_path = posixpath.join(EXPERIMENT_DIR, "preflight_channel_scan.json")
if os.path.exists(_scan_path) and STAGE34_FINGERPRINT == STAGE34_FINGERPRINT_INITIAL:
    with open(_scan_path) as _fh:
        _scan = json.load(_fh)
else:
    _mins, _maxs, _sum, _n = np.full(8, np.inf), np.full(8, -np.inf), np.zeros(8), 0
    for _i, _e in enumerate(DATA["train"] + DATA["val"]):
        _x = cached_sample(_e)["stage5_input"]
        _mins = np.minimum(_mins, _x.reshape(-1, 8).min(0))
        _maxs = np.maximum(_maxs, _x.reshape(-1, 8).max(0))
        if _i < len(DATA["train"]):
            _sum += _x.reshape(-1, 8).mean(0)
            _n += 1
        if (_i + 1) % 500 == 0:
            print(f"    channel scan {_i + 1}/{len(DATA['train']) + len(DATA['val'])}")
    _scan = {"min": _mins.tolist(), "max": _maxs.tolist(), "train_mean": (_sum / _n).tolist()}
    with open(_scan_path, "w") as _fh:
        json.dump(_scan, _fh, indent=2)
check(9, "Cached channels within [0,1]", min(_scan["min"]) >= 0.0 and max(_scan["max"]) <= 1.0,
      "min " + str([round(v, 4) for v in _scan["min"]]) + "; max " + str([round(v, 4) for v in _scan["max"]])
      + "; train means (descriptive) " + str([round(v, 5) for v in _scan["train_mean"]]))
check(10, "Split hash and population", SPLIT_SHA256 == msr.EXPECTED_SPLIT_SHA256
      and (len(DATA["train"]), len(DATA["val"])) == (EXPECTED_TRAIN_N, EXPECTED_VAL_N),
      f"{SPLIT_SHA256[:12]}..., {len(DATA['train'])} / {len(DATA['val'])}")
check(11, "Stage 3/4 checkpoints and caches unchanged", STAGE34_FINGERPRINT == STAGE34_FINGERPRINT_INITIAL,
      STAGE34_FINGERPRINT)
check(12, "Identical CORN head initialisation for P and PL", _ok12, "GlorotUniform(seed) + zeros, 3 seeds")

_wd = {}
for _arm in ("P", "PL"):
    fresh_session()                    # clear_session() resets the global dtype policy to float32 ...
    enable_mixed_precision(True)       # ... so mixed precision is enabled AFTER it, as build_model() does
    _m = pl.compile_pl_model(pl.build_pl_model(_arm, 42, REFERENCE_ARRAYS), CLASS_WEIGHTS, LEARNING_RATE,
                             WEIGHT_DECAY)
    _r = pl.weight_decay_report(_m)
    _r["policies"] = sorted(model_precision_policies(_m))
    _r["policy_ok"] = precision_is_consistent(expected_policy_name(True), model_precision_policies(_m))
    _r["optimizer"] = type(_m.optimizer).__name__
    _r["loss_scale_ok"] = getattr(_m.optimizer, "inner_optimizer", None) is not None
    _wd[_arm] = _r
    del _m
check(13, "Weight-decay exclusions (1-D incl. LayerScale) and mixed precision",
      all(r["ok"] and r["policy_ok"] and r["loss_scale_ok"] for r in _wd.values()),
      {a: f"wd_ok={r['ok']} ({r['one_d']} 1-D exempt, LayerScale {r['layer_scale_exempt']}/{r['layer_scale']}, "
          f"{len(r['decayed_one_d'])} 1-D decayed, {len(r['exempt_multi_d'])} >=2-D exempt); "
          f"policy_ok={r['policy_ok']} {r['policies']}; loss_scale_ok={r['loss_scale_ok']} ({r['optimizer']})"
       for a, r in _wd.items()})
fresh_session()

with open(posixpath.join(EXPERIMENT_DIR, f"preflight_{datetime.datetime.now():%Y-%m-%d_%H-%M-%S}.json"), "w") as _fh:
    json.dump({"commit": REPO_COMMIT, "weights": WEIGHTS, "checks": PREFLIGHT}, _fh, indent=2, default=str)
_failed = [k for k, v in PREFLIGHT.items() if not v["ok"]]
if _failed:
    raise RuntimeError(f"PRE-FLIGHT FAILED -- nothing is trained: {_failed}")
print("\nAll 13 pre-flight assertions PASSED.")

In [ ]:
# ==== [5] MANIFEST AND RUN STATUS (derived from disk every time) ====
MANIFEST = [{"arm": a, "seed": s, "label": f"{a}-{s}",
             "dir": posixpath.join(EXPERIMENT_DIR, pl.ARMS[a]["dirname"], f"seed_{s}")} for a, s in pl.RUN_ORDER]
for _i, _run in enumerate(MANIFEST, start=1):
    _run["index"] = _i


def reference_dir(seed):
    return msr.run_dir(SIX_RUN_ROOT, SIX_RUN_ID, "NO_RACAF", seed)


def per_sample_valid(path):
    if not os.path.exists(path):
        return False
    return len(pd.read_csv(path, usecols=["image_id", "true_grade", "p_gt_2"])) == EXPECTED_VAL_N


def evaluated(run_dir):
    ev = posixpath.join(run_dir, "evaluation")
    return all(per_sample_valid(posixpath.join(ev, f"per_sample_{w}.csv")) for w in ("best", "last"))


def run_status(run):
    if msr.read_stop_decision(run["dir"]) is not None:
        return "COMPLETED" if evaluated(run["dir"]) else "NEEDS_EVALUATION"
    if ckpt.checkpoint_evidence(posixpath.join(run["dir"], "checkpoints")) or msr.read_history(run["dir"]):
        return "INTERRUPTED"
    return "NOT_STARTED"


def write_status_table():
    table = [{"index": r["index"], "label": r["label"], "status": run_status(r), "dir": r["dir"]} for r in MANIFEST]
    with open(posixpath.join(EXPERIMENT_DIR, "run_status.json"), "w") as fh:
        json.dump({"updated": datetime.datetime.now().isoformat(timespec="seconds"), "runs": table}, fh, indent=2)
    return table


for _row in write_status_table():
    print(f"{_row['index']}. {_row['label']:<8} {_row['status']:<18} {_row['dir']}")

In [ ]:
# ==== [6] MODEL CONSTRUCTION AND EVALUATION ====
def build_model(seed, arm):
    enable_mixed_precision(True)
    model = pl.compile_pl_model(pl.build_pl_model(arm, seed, REFERENCE_ARRAYS), CLASS_WEIGHTS,
                                LEARNING_RATE, WEIGHT_DECAY)
    assert precision_is_consistent(expected_policy_name(True), model_precision_policies(model))
    assert getattr(model.optimizer, "inner_optimizer", None) is not None, "no LossScaleOptimizer"
    assert pl.weight_decay_report(model)["ok"]
    return model


def write_evaluation(run_dir, which, rows, extra):
    ev = posixpath.join(run_dir, "evaluation")
    os.makedirs(ev, exist_ok=True)
    with open(posixpath.join(ev, f"per_sample_{which}.csv"), "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    metrics = pl.run_metrics(pd.DataFrame(rows))
    metrics.update(extra)
    with open(posixpath.join(ev, f"metrics_{which}.json"), "w") as fh:
        json.dump(metrics, fh, indent=2, default=float)
    return metrics


def evaluate_weights(seed, arm, weights_path):
    fresh_session()
    model = build_model(seed, arm)
    ckpt.load_model_weights_only(model, weights_path)
    rows = msr.evaluate_arm_from_disk(model, DATA["val"], cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
    del model
    return rows


def evaluate_new_run(run):
    slot_dir, pointer = msr.read_best(run["dir"])
    assert slot_dir is not None, f"{run['label']}: no BEST published"
    rows = evaluate_weights(run["seed"], run["arm"], os.path.join(slot_dir, ckpt.MODEL_WEIGHTS_FILENAME))
    m = write_evaluation(run["dir"], "best", rows, {"label": run["label"], "checkpoint": "BEST",
                                                    "best_epoch_index_0based": pointer["epoch"]})
    last_dir = ckpt.find_resumable_generation(posixpath.join(run["dir"], "checkpoints"), verbose=False)
    assert last_dir is not None, f"{run['label']}: no valid LAST generation"
    rows = evaluate_weights(run["seed"], run["arm"], os.path.join(last_dir, ckpt.MODEL_WEIGHTS_FILENAME))
    write_evaluation(run["dir"], "last", rows, {"label": run["label"], "checkpoint": "LAST",
                                                "generation": os.path.basename(last_dir)})
    print(f"  evaluated {run['label']}: AUROC P(>=3) g4 vs 0-2 = {m['auroc_ge3_g4_vs_g012']:.4f}, "
          f"QWK = {m['qwk']:.4f}")

In [ ]:
# ==== [7] AUTOMATIC RUN EXECUTION (six runs, each exactly once; interrupted runs resume) ====
def acquire_lock_waiting(run_dir):
    while True:
        try:
            msr.acquire_lock(run_dir, owner_id=msr.OWNER_ID)
            return
        except msr.RunLockedError as error:
            print(f"  lock held by another runtime; waiting for it to go stale. ({error})")
            time.sleep(60)


def arm_config_mapping(arm, seed):
    return {"protocol_version": SIX_RUN_PREREG["protocol_version"] + "-pl-convnext",
            "experiment_id": EXPERIMENT_ID, "arm": f"CONVNEXT_TINY_{arm}", "run_seed": seed,
            "split_seed": msr.SPLIT_SEED, "split_sha256": SPLIT_SHA256, "model": f"convnext_tiny_{arm.lower()}",
            "input_channels": pl.ARMS[arm]["channels"], "weights_sha256": pl.WEIGHTS_SHA256,
            "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
            "weight_decay_exclude": ["all 1-D"] + list(msr.WEIGHT_DECAY_EXCLUDE_NAMES),
            "class_weights": list(CLASS_WEIGHTS), "monitor": MONITOR, "mode": MODE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE, "reduce_lr_patience": REDUCE_LR_PATIENCE,
            "reduce_lr_factor": REDUCE_LR_FACTOR, "min_lr": MIN_LR}


def train_arm_run(run):
    run_dir, seed, arm = run["dir"], run["seed"], run["arm"]
    msr.ensure_run_dir(run_dir)
    mapping = arm_config_mapping(arm, seed)
    config_hash = ckpt.config_hash(mapping)
    manifest_path = posixpath.join(run_dir, msr.RUN_MANIFEST_FILENAME)
    if os.path.exists(manifest_path):
        msr.verify_run_manifest(run_dir, mapping, config_hash)
    else:
        msr.write_run_manifest(manifest_path, mapping, config_hash, repo_dir=colab_config.REPO_DIR)
    sealed = msr._sealed_stop(posixpath.join(run_dir, "checkpoints"), MAX_EPOCHS)
    if sealed is not None:
        msr.write_stop_decision(run_dir, sealed[0], sealed[1])
        return
    fresh_session()
    model = build_model(seed, arm)
    acquire_lock_waiting(run_dir)
    try:
        sources = {}
        for relative_path, symbols in msr.TRAINING_BEHAVIOR_SOURCES + PL_MODULE_SOURCES:
            sources.update(msr.normalized_source_digests(relative_path, symbols, msr.REPO_ROOT))
        behaviour_config = {k: v for k, v in mapping.items() if k not in ("experiment_id", "protocol_version")}
        behaviour_config["mixed_precision"] = True
        components = {"fingerprint_version": msr.TRAINING_BEHAVIOR_FINGERPRINT_VERSION,
                      "configuration": behaviour_config,
                      "train_population": msr.population_digest(DATA["train"]),
                      "validation_population": msr.population_digest(DATA["val"]), "sources": sources}
        msr.reconcile_training_behavior(
            run_dir, {"training_behavior_hash": msr._canonical_hash(components), "components": components},
            msr._current_git_commit(colab_config.REPO_DIR), verbose=1)
        trainer = Trainer(TrainingConfig(
            run_dir=run_dir, epochs=MAX_EPOCHS, monitor=MONITOR, mode=MODE, mixed_precision=True, resume=True,
            early_stopping_patience=EARLY_STOPPING_PATIENCE, reduce_lr_patience=REDUCE_LR_PATIENCE,
            reduce_lr_factor=REDUCE_LR_FACTOR, min_lr=MIN_LR, precision_check="error",
            repo_dir=colab_config.REPO_DIR,
            checkpoint_options=CheckpointOptions(
                experiment_id=f"{pl.ARMS[arm]['dirname']}/seed_{seed}", config_hash=config_hash,
                dataset_version="aptos2019-joint-cache-v1", staging_dir=STAGING_DIR, keep_generations=2,
                verbose=1)))
        trainer.prepare(model)
        initial_epoch = trainer.resolve_initial_epoch()
        if initial_epoch > 0:
            trainer.restore(model)
            print(f"  resumed {run['label']} at epoch {initial_epoch}")
        if initial_epoch >= MAX_EPOCHS:
            msr.write_stop_decision(run_dir, initial_epoch, "epoch_cap")
            return
        early = next(c for c in trainer.callbacks if isinstance(c, tf.keras.callbacks.EarlyStopping))
        state_callback = next(c for c in trainer.callbacks if isinstance(c, TrainingStateCheckpoint))
        val_ds = itd.make_epoch_dataset(DATA["val"], epoch=0, run_seed=seed, cache_dir=LOCAL_CACHE_DIR,
                                        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR, batch_size=BATCH_SIZE,
                                        augment=False)
        full_history_dir = posixpath.join(run_dir, "history_full")
        os.makedirs(full_history_dir, exist_ok=True)
        for epoch in range(initial_epoch, MAX_EPOCHS):
            msr.heartbeat_lock(run_dir, owner_id=msr.OWNER_ID)
            train_ds = itd.make_epoch_dataset(DATA["train"], epoch=epoch, run_seed=seed, cache_dir=LOCAL_CACHE_DIR,
                                              racaf_cache_dir=LOCAL_RACAF_CACHE_DIR, batch_size=BATCH_SIZE,
                                              augment=True)
            model.fit(train_ds, validation_data=val_ds, epochs=epoch + 1, initial_epoch=epoch,
                      callbacks=trainer.callbacks, verbose=1)
            generation = state_callback.last_generation_dir
            assert generation is not None, f"epoch {epoch}: no checkpoint generation written"
            state = ckpt.read_state(generation)
            msr.write_epoch_history(run_dir, state)
            with open(posixpath.join(full_history_dir, f"epoch_{state.completed_epoch:04d}.json"), "w") as fh:
                json.dump(dict((state.extra or {}).get("epoch_logs") or {}, epoch=state.completed_epoch,
                               learning_rate=state.learning_rate), fh, indent=2)
            if state.best_epoch == epoch:
                msr.publish_best(run_dir, generation, state, repo_dir=colab_config.REPO_DIR, verbose=1)
            if early.stopped_epoch:
                msr.write_stop_decision(run_dir, state.completed_epoch, "early_stopping")
                break
            if state.completed_epoch >= MAX_EPOCHS:
                msr.write_stop_decision(run_dir, state.completed_epoch, "epoch_cap")
                break
    finally:
        msr.release_lock(run_dir, owner_id=msr.OWNER_ID)
        del model


for run in MANIFEST:
    status = run_status(run)
    if status == "COMPLETED":
        continue
    print(f"\n=== [{run['index']}/{len(MANIFEST)}] {run['label']} ({status}) ===")
    if status in ("NOT_STARTED", "INTERRUPTED"):
        train_arm_run(run)
    if msr.read_stop_decision(run["dir"]) is not None:
        evaluate_new_run(run)
    write_status_table()
    print(f"  {run['label']}: {run_status(run)}")

print("\nRun status after this session:")
for _row in write_status_table():
    print(f"{_row['index']}. {_row['label']:<8} {_row['status']}")

In [ ]:
# ==== [8] LOCKED COMPARISON, VERDICTS AND FINAL REPORT ====
DONE = [r for r in MANIFEST if run_status(r) == "COMPLETED"]
FINAL = len(DONE) == len(MANIFEST)
VAL_IDS = [i for i, _ in DATA["val"]]
Y_VAL = np.array([int(g) for _, g in DATA["val"]])
SCALAR = ["auroc_ge3_g4_vs_g012", "auroc_ge3_g3_vs_g012", "auroc_p4_g4_vs_g012",
          "grade4_sensitivity_at_95_specificity", "qwk", "mae", "grade3_recall", "false_urgent_rate",
          "grade4_recall", "grade4_precision", "grade4_decoded_le2"]


def frame_for(path):
    frame = pd.read_csv(path, dtype={"image_id": str}).set_index("image_id").loc[VAL_IDS].reset_index()
    assert (frame["true_grade"].to_numpy() == Y_VAL).all(), f"{path}: membership/grades differ"
    return frame


FRAMES, METRICS = {}, {}
for run in DONE:
    for which in ("best", "last"):
        f = frame_for(posixpath.join(run["dir"], "evaluation", f"per_sample_{which}.csv"))
        FRAMES[(run["arm"], run["seed"], which)] = f
        METRICS[(run["arm"], run["seed"], which)] = pl.run_metrics(f)
for seed in pl.SEEDS:
    f = frame_for(posixpath.join(reference_dir(seed), "evaluation", "per_sample_best.csv"))
    FRAMES[("REF", seed, "best")] = f
    METRICS[("REF", seed, "best")] = pl.run_metrics(f)

L = [f"# P/PL — pretrained ConvNeXt-Tiny ± frozen Stage 3/4 priors — {'FINAL' if FINAL else 'PARTIAL'} report", "",
     f"Experiment `{EXPERIMENT_DIR}` · commit `{REPO_COMMIT}` · runs complete {len(DONE)}/6", "",
     "## 1. Colab / runtime verification", "",
     f"- GPUs {[g.name for g in GPUS]}; keras {keras.__version__}; tensorflow {tf.__version__}",
     f"- Weights `{WEIGHTS['source_url']}` sha256 `{WEIGHTS['sha256']}` ({WEIGHTS['obtained_via']}; cached at "
     f"`{WEIGHTS['drive_cache']}`)", "",
     "## 2. Pre-flight", ""] + [f"- {'PASS' if v['ok'] else 'FAIL'} {k}: {v['detail']}" for k, v in PREFLIGHT.items()]

L += ["", "## 3. Six-run results (BEST; AUROC = P(grade>=3), grade 4 vs 0-2)", "",
      "| run | AUROC | sens@95%spec | QWK | MAE | grade-3 recall | false-urgent | BEST epoch |", "|---|---|---|---|---|---|---|---|"]
for run in MANIFEST:
    m = METRICS.get((run["arm"], run["seed"], "best"))
    if m is None:
        L.append(f"| {run['label']} | {run_status(run)} | | | | | | |")
        continue
    ep = json.load(open(posixpath.join(run["dir"], "evaluation", "metrics_best.json"))).get("best_epoch_index_0based")
    L.append(f"| {run['label']} | {m['auroc_ge3_g4_vs_g012']:.4f} | {m['grade4_sensitivity_at_95_specificity']:.3f} | "
             f"{m['qwk']:.4f} | {m['mae']:.4f} | {m['grade3_recall']:.3f} | {m['false_urgent_rate']:.3f} | {ep} |")
for seed in pl.SEEDS:
    m = METRICS[("REF", seed, "best")]
    L.append(f"| NO_RACAF-{seed} (frozen ref) | {m['auroc_ge3_g4_vs_g012']:.4f} | "
             f"{m['grade4_sensitivity_at_95_specificity']:.3f} | {m['qwk']:.4f} | {m['mae']:.4f} | "
             f"{m['grade3_recall']:.3f} | {m['false_urgent_rate']:.3f} | - |")

RESULTS = {"final": FINAL, "experiment_dir": EXPERIMENT_DIR, "commit": REPO_COMMIT, "weights": WEIGHTS,
           "preflight": PREFLIGHT, "metrics": {f"{a}-{s}-{w}": v for (a, s, w), v in METRICS.items()}}
if FINAL:
    for arm in ("P", "PL"):
        vals = np.array([METRICS[(arm, s, "best")]["auroc_ge3_g4_vs_g012"] for s in pl.SEEDS])
        L.append(f"\n{arm} mean ± SD AUROC: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}")
    DECISIONS = {}
    for tag, base in (("A", "P"), ("B", "REF")):
        deltas = {k: np.array([METRICS[("PL", s, "best")][k] - METRICS[(base, s, "best")][k] for s in pl.SEEDS])
                  for k in SCALAR}
        guard = {k: float(deltas[k].mean()) for k in pl.GUARDRAILS}
        decision = pl.decide(deltas["auroc_ge3_g4_vs_g012"], guard)
        decision["bootstrap"] = c1.paired_bootstrap_primary(
            Y_VAL, [FRAMES[(base, s, "best")]["p_gt_2"].to_numpy() for s in pl.SEEDS],
            [FRAMES[("PL", s, "best")]["p_gt_2"].to_numpy() for s in pl.SEEDS])
        decision["secondary_mean_deltas"] = {k: float(v.mean()) for k, v in deltas.items()}
        decision["secondary_per_seed_deltas"] = {k: v.tolist() for k, v in deltas.items()}
        DECISIONS[tag] = decision
    RESULTS["decisions"] = DECISIONS
    for tag, title in (("A", "PL − P (primary, controlled)"), ("B", "PL − frozen NO_RACAF (system-level only)")):
        d = DECISIONS[tag]
        b = d["bootstrap"]
        L += ["", f"## {4 if tag == 'A' else 5}. {title}", "",
              "Per-seed Δ AUROC: " + ", ".join(f"{s}: {v:+.4f} (95% CI {c[0]:+.3f} to {c[1]:+.3f})"
                                              for s, v, c in zip(pl.SEEDS, d["per_seed_delta"], b["per_seed_ci"])),
              f"Mean Δ {d['mean_delta']:+.4f} (SD {d['sd_delta']:.4f}); seed-mean bootstrap 95% CI "
              f"{b['mean_delta_ci'][0]:+.4f} to {b['mean_delta_ci'][1]:+.4f}; positive in {d['seeds_positive']}/3",
              "Secondary mean Δ: " + ", ".join(f"{k} {v:+.4f}" for k, v in d["secondary_mean_deltas"].items()),
              "Guardrails: " + ", ".join(f"{k} {g['mean_delta']:+.4f} ({'holds' if g['holds'] else 'FAILS'})"
                                         for k, g in d["guardrails"].items()),
              f"**Pre-registered verdict {tag}: {d['verdict']}**"]
    L += ["", "## 6. BEST vs LAST (AUROC / QWK)", "", "| run | BEST AUROC | LAST AUROC | BEST QWK | LAST QWK |",
          "|---|---|---|---|---|"]
    for run in MANIFEST:
        bm, lm = METRICS[(run["arm"], run["seed"], "best")], METRICS[(run["arm"], run["seed"], "last")]
        L.append(f"| {run['label']} | {bm['auroc_ge3_g4_vs_g012']:.4f} | {lm['auroc_ge3_g4_vs_g012']:.4f} | "
                 f"{bm['qwk']:.4f} | {lm['qwk']:.4f} |")
    _fp_now = pl.directory_fingerprint(STAGE34_PATHS)
    RESULTS["stage34_unchanged"] = _fp_now == STAGE34_FINGERPRINT_INITIAL
    L += ["", "## 7. Interpretation limits", "",
          "- A tests only whether the frozen priors add information to the same pretrained ConvNeXt-T on APTOS "
          "development data. B is system-level and does not isolate pretraining; neither shows that random "
          "initialisation caused earlier failures. P/PL is not a novel architecture. DDR not used.",
          f"- Stage 3/4 models and caches unchanged after the runs: {RESULTS['stage34_unchanged']}",
          "", "## 8. Protocol deviations", "",
          "No protocol deviations." if RESULTS["stage34_unchanged"] else "Stage 3/4 fingerprint changed -- investigate."]
    pd.DataFrame([dict(run=f"{a}-{s}", checkpoint=w, **{k: v for k, v in m.items() if k in SCALAR})
                  for (a, s, w), m in METRICS.items()]).to_csv(posixpath.join(EXPERIMENT_DIR, "per_run_results.csv"),
                                                               index=False)
report = "\n".join(L)
with open(posixpath.join(EXPERIMENT_DIR, "REPORT.md"), "w", encoding="utf-8") as fh:
    fh.write(report)
_tmp = posixpath.join(EXPERIMENT_DIR, "results.json.tmp")
with open(_tmp, "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=float)
os.replace(_tmp, posixpath.join(EXPERIMENT_DIR, "results.json" if FINAL else "results_partial.json"))
print(report)
print("\nOutputs:", EXPERIMENT_DIR)